# Capstone: enterprise and research question answering: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the cells download two small models, all-MiniLM-L6-v2 and cross-encoder/ms-marco-MiniLM-L-6-v2 (about 90 MB each; internet but no GPU), and chromadb (about 100 MB of packages). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install chromadb==1.5.9 sentence-transformers==6.3.0

## Capstone A: the specification of a help assistant

chromadb with the permission rule applied before the search, using MiniLM embeddings.

In [ ]:
docs = {
    "exp-travel": {"text": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-sso": {"text": "Single sign-on with SAML is available on the Business plan and above.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-carry": {"text": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "2026-12-31", "deleted": False},
    "faq-seats": {"text": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-contract": {"text": "Acme Freight contract: 40 Business seats, renewal on 1 October, discount 12 percent.", "tenant": "acme", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-merge": {"text": "Two tickets from the same customer can be merged. The older ticket keeps its number.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-hours": {"text": "Acme Freight support is open from 08:00 to 20:00 on working days.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err4012": {"text": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-contract": {"text": "Birch Clinics contract: 12 Team seats, renewal on 1 March, discount 5 percent.", "tenant": "birch", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "hr-salary": {"text": "Salary bands: grade 3 starts at 900000 rupees a year and grade 4 at 1300000 rupees.", "tenant": "orbit", "allow": ["hr"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-carry-old": {"text": "Up to 10 unused annual leave days may be carried over and must be used by 30 June.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2023-01-01", "valid_to": "2025-03-31", "deleted": False},
    "sec-incident": {"text": "Incident note: an access key was exposed on the build server and revoked the same day.", "tenant": "orbit", "allow": ["security"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
}
users = {
    "asha": {"tenant": "orbit", "groups": ["staff"]},
    "acme-admin": {"tenant": "acme", "groups": ["agent", "admin"]},
    "acme-agent": {"tenant": "acme", "groups": ["agent"]},
}

def allowed(user, doc, as_of="2026-06-01"):
    """True only if every rule passes. A record with a missing field is refused (deny by default)."""
    if doc.get("deleted", True):
        return False
    if doc.get("tenant") not in (user["tenant"], "shared"):
        return False
    if not set(doc.get("allow", [])) & (set(user["groups"]) | {"everyone"}):
        return False
    return doc.get("valid_from", "9999") <= as_of <= doc.get("valid_to", "0000")

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

client = chromadb.EphemeralClient()                    # in memory, nothing is written to disk
col = client.create_collection("spec-check", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
col.add(ids=list(docs), embeddings=embed([d["text"] for d in docs.values()]),
        metadatas=[{"rid": i} for i in docs])

for user, question in [("asha", "can I fly business class"), ("acme-admin", "how many seats does our contract include"), ("acme-agent", "when does the Birch Clinics contract renew")]:
    ids = [i for i, d in docs.items() if allowed(users[user], d)]     # rules first
    top = col.query(query_embeddings=embed([question]), n_results=3, where={"rid": {"$in": ids}})["ids"][0]
    print(user, "|", question)
    print("  ", top)

asha | can I fly business class
   ['exp-travel', 'faq-sso', 'leave-carry']
acme-admin | how many seats does our contract include
   ['faq-seats', 'acme-contract', 'kb-merge']
acme-agent | when does the Birch Clinics contract renew
   ['faq-sso', 'acme-hours', 'kb-err4012']


## Capstone B: the specification of research question answering

MiniLM ranking of the sources for the customer satisfaction question.

In [ ]:
sources = {
    "s01": ("study", 2025, "Trial A: in 240 support agents, suggested replies cut average handling time by 12 percent over 8 weeks."),
    "s02": ("study", 2025, "Trial A also measured customer satisfaction and found no clear change over the 8 weeks."),
    "s03": ("study", 2026, "Trial B: in 90 support agents, suggested replies cut average handling time by 4 percent over 12 weeks."),
    "s04": ("blog", 2024, "A vendor blog says suggested replies cut handling time by 40 percent, without describing how it was measured."),
    "s05": ("study", 2026, "Trial B found that new agents gained the most, while agents with over two years of service gained almost nothing."),
    "s06": ("report", 2025, "An internal report counted that agents accepted 38 percent of suggested replies and edited most of them."),
    "s07": ("study", 2025, "A field study of 60 agents saw more reopened tickets when agents sent suggestions without editing them."),
    "s08": ("report", 2026, "A cost review put the price of the suggestion service at 7 credits per agent per month."),
    "s09": ("note", 2024, "Meeting note: the team hopes suggested replies will also help with training new agents."),
    "s10": ("report", 2025, "A privacy review requires that customer names and addresses are removed before text is sent to the model."),
    "s11": ("study", 2024, "A laboratory test with 30 students found faster writing of replies when a draft was offered."),
    "s12": ("report", 2026, "The support team has 120 agents, of whom 45 joined within the last year."),
    "s13": ("study", 2025, "A study of chat support found that suggestions in a second language reduced spelling mistakes."),
    "s14": ("blog", 2026, "A blog post lists ten tips for writing friendly replies to angry customers."),
    "s15": ("report", 2024, "The ticket system keeps the text of closed tickets for 24 months."),
    "s16": ("note", 2026, "Meeting note: the trial in our own team has not started yet."),
}

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(sources)
vectors = model.encode([sources[i][2] for i in ids], normalize_embeddings=True, show_progress_bar=False)
question = model.encode("do suggested replies change customer satisfaction", normalize_embeddings=True)
order = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(vectors[j] @ question), ids[j]))]
print("top 5:", order[:5])
print("rank of s02:", order.index("s02") + 1)

top 5: ['s14', 's06', 's01', 's09', 's02']
rank of s02: 5
